# LLM Prompting Lab

A practical notebook for experimenting with prompt-engineering techniques using the OpenAI API.

The examples focus on how instruction style, examples, role, context, decomposition, and reasoning-oriented prompt patterns can influence model responses.

> Some advanced patterns here (for example, ReAct, self-consistency, and tree-of-thought) are demonstrated at the prompt level. They are not full agent/tool implementations.

## Setup

The notebook reads the OpenAI API key from the environment when available. If it is not set, it asks for the key securely at runtime so the key is not stored in the notebook.

In [ ]:
%pip install -q openai

In [ ]:
import os
import json
import datetime
import pathlib
from getpass import getpass

MODEL = "gpt-4o-mini"
GEN_ARGS = {"temperature": 0.2}
OUT_DIR = pathlib.Path("prompt_runs")
OUT_DIR.mkdir(exist_ok=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY") or getpass("OpenAI API key: ")

def log_result(kind, name, prompt, output):
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    (OUT_DIR / f"{kind}_{name}_{ts}.prompt.txt").write_text(prompt, encoding="utf-8")
    (OUT_DIR / f"{kind}_{name}_{ts}.output.txt").write_text(output, encoding="utf-8")
    print(f"Logged → {OUT_DIR}")

In [ ]:
from openai import OpenAI

_openai = OpenAI(api_key=OPENAI_API_KEY)

def llm_openai(prompt, model=MODEL, system=None, **gen_args):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = _openai.chat.completions.create(
        model=model,
        messages=messages,
        **gen_args
    )
    return response.choices[0].message.content

def run(prompt, system=None, **gen_args):
    return llm_openai(prompt, system=system, **gen_args)

1) Zero‑Shot Prompting
How: Give clear instructions without examples.

When: Use for simple, unambiguous tasks (translations, factual queries).

In [ ]:
# Run: zero-shot
PROMPT = r"""Translate the English phrase 'Flowers on the road' to Spanish."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("zero-shot", "example", PROMPT, out)


How: Give clear instructions and add a single example to demonstrate the task/output.

When: Use when one example is enough to clarify the task or format.

Example prompt

In uppercase, return the Spanish translation of the English word 'basket' only.
EXAMPLE:
English word (input): River
Spanish translation (output): RÍO

In [ ]:
# Run: one-shot
PROMPT = r"""In uppercase, return the Spanish translation of the English word 'basket' only.
EXAMPLE:
English word (input): River
Spanish translation (output): RÍO"""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("one-shot", "example", PROMPT, out)

How: Give clear instructions with a few examples in the prompt.

When: Use to adapt the model to a task/domain without fine‑tuning; improves consistency.

Example prompt

Return the sentiment of this statement 'The lecture was quite boring' only. Either Positive, Negative, or Neutral.
EXAMPLES:
'This movie was great!': Positive
'I hated the service.': Negative
'I don't know how I feel about it.': Neutral

In [ ]:
# Run: few-shot
PROMPT = r"""Return the sentiment of this statement 'The lecture was quite boring' only. Either Positive, Negative, or Neutral.
EXAMPLES:
'This movie was great!': Positive
'I hated the service.': Negative
'I don't know how I feel about it.': Neutral"""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("few-shot", "example", PROMPT, out)


4) Role Prompting
How: Give clear instructions and assign a persona to steer tone and perspective.

When: Use for open‑ended tasks where tone or viewpoint matters.
Example prompt

ROLE: Act as a sweet college girl who uses Gen‑Z slang.
TASK: Write a short blog (≈500 words) with 4 points about college hacks.

In [ ]:
# Run: role
PROMPT = r"""ROLE: Act as a sweet college girl who uses Gen‑Z slang.
TASK: Write a short blog (≈500 words) with 4 points about college hacks."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("role", "example", PROMPT, out)


5) Style Prompting
How: Specify desired style, tone, or genre explicitly.

When: Use when the output must match a particular style or tone.

Example prompt

Write a brief formal email requesting a raise.

In [ ]:
# Run: style
PROMPT = r"""Write a brief formal email requesting a raise."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("style", "example", PROMPT, out)

6) Emotion Prompting
How: Add an emotionally charged phrase/sentence to shape affect.

When: Use in creative writing (stories, poetry) where emotion matters.

Example prompt

Write a poem about my lost imaginary friend who never gave up. I still miss my friend.

In [ ]:
# Run: emotion
PROMPT = r"""Write a poem about my lost imaginary friend who never gave up. I still miss my friend."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("emotion", "example", PROMPT, out)

7) Contextual Prompting
How: Provide background/custom info before instructions.

When: Use when domain‑specific context is needed (RAG/chatbots).

Example prompt

CONTEXT: My name is Jennifer Luke and I'm a marketing manager at JL Firm.
Write an email to the team about the upcoming campaign.

In [ ]:
# Run: contextual
PROMPT = r"""CONTEXT: My name is Jennifer Luke and I'm a marketing manager at JL Firm.
Write an email to the team about the upcoming campaign."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("contextual", "example", PROMPT, out)


8) Rephrase‑and‑Respond (RaR)
How: Tell the model to rewrite the question as a better prompt, then answer it.

When: Use for complex tasks to improve understanding and accuracy.

Example prompt

Rephrase and expand the following question, then answer it: What is the difference between correlation and causation?

In [ ]:
# Run: rephrase-and-respond
PROMPT = r"""Rephrase and expand the following question, then answer it: What is the difference between correlation and causation?"""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("rephrase-and-respond", "example", PROMPT, out)


9) Re‑Reading (RE2)
How: Ask the question, add 'Read that again:' and repeat the question verbatim.

When: Use for reasoning problems that benefit from refocus.

Example prompt

A farmer has a rectangular field that is 3× as long as it is wide. The perimeter is 400 m. What are the dimensions?
Read that again: "A farmer has a rectangular field that is 3× as long as it is wide. The perimeter is 400 m. What are the dimensions?"

In [ ]:
# Run: re-reading
PROMPT = r"""A farmer has a rectangular field that is 3× as long as it is wide. The perimeter is 400 m. What are the dimensions?
Read that again: "A farmer has a rectangular field that is 3× as long as it is wide. The perimeter is 400 m. What are the dimensions?"""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("re-reading", "example", PROMPT, out)


10) System Prompting
How: Set high-level behavior or tone in a real system message.

When: Use to govern assistant behavior across a conversation.

Example: Set a concise, factual assistant behavior and ask it to explain a concept.

In [ ]:
# Run: system prompting
SYSTEM = "You are a helpful assistant that provides factual, concise responses."
PROMPT = "Explain the difference between supervised and unsupervised learning in two sentences."
out = run(PROMPT, system=SYSTEM, **GEN_ARGS)
print(out)
log_result("system", "example", PROMPT, out)

11) Self‑Ask
How: Tell the model to decompose the question into sub‑questions and answer each.

When: Use for multi‑step reasoning and decision‑making tasks.

Example prompt

Should I pursue a master's in data science? Break this into sub‑questions, answer them, then provide a final recommendation.

In [ ]:
# Run: self-ask
PROMPT = r"""Should I pursue a master's in data science? Break this into sub‑questions, answer them, then provide a final recommendation."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("self-ask", "example", PROMPT, out)


12) Chain‑of‑Thought (CoT)
How: Add a reasoning nudge such as 'Let's think step by step.'

When: Use for math/logic problems. Prefer brief step outlines over exposing hidden chain‑of‑thought in production.

Example prompt

What is the total cost of a meal with a 10% discount and 7% tax? Let's think step by step.

In [ ]:
# Run: chain-of-thought
PROMPT = r"""What is the total cost of a meal with a 10% discount and 7% tax? Let's think step by step."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("chain-of-thought", "example", PROMPT, out)


13) Step‑Back Prompting
How: Ask a broad question, then a targeted one based on the broader answer.

When: Use for analysis/decisions that depend on multiple factors.

Example prompt

Explain key factors that influence expansion to a new market. Based on this, should a tech company expand to Europe?

In [ ]:
# Run: step-back
PROMPT = r"""Explain key factors that influence expansion to a new market. Based on this, should a tech company expand to Europe?"""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("step-back", "example", PROMPT, out)


14) Self‑Consistency
How: Ask for multiple candidate answers; return the most frequent.

When: Use when many answers are possible and you want higher reliability.

Example prompt

What is the most popular programming language for machine learning? Generate 5 answers and return the one that appears most often.

In [ ]:
# Run: self-consistency
PROMPT = r"""What is the most popular programming language for machine learning? Generate 5 answers and return the one that appears most often."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("self-consistency", "example", PROMPT, out)

15) Thread‑of‑Thought (ThoT)
How: Ask to 'walk through in manageable parts step by step' (briefly).

When: Use for Q&A over large, complex contexts (e.g., RAG).

Example prompt

CONTEXT: Party music preferences problem (10 guests, 3 genres, limited speakers).
Walk me through in manageable parts to maximize satisfied guests.

In [ ]:
# Run: thread-of-thought
PROMPT = r"""CONTEXT: Party music preferences problem (10 guests, 3 genres, limited speakers).
Walk me through in manageable parts to maximize satisfied guests."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("thread-of-thought", "example", PROMPT, out)

16) Tree‑of‑Thought (ToT)
How: At each step, generate multiple options, evaluate them, pick the best, continue.

When: Use for deep reasoning, multi‑step planning, high accuracy.

Example prompt

Design a coffee cup that keeps drinks hot longer. Break down steps; at each, generate options, evaluate (feasibility/cost/impact), choose best, iterate to a final design.

In [ ]:
# Run: tree-of-thought
PROMPT = r"""Design a coffee cup that keeps drinks hot longer. Break down steps; at each, generate options, evaluate (feasibility/cost/impact), choose best, iterate to a final design."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("tree-of-thought", "example", PROMPT, out)

17) ReAct-Style Prompting (Conceptual)
How: ReAct normally alternates reasoning, tool actions, and observations.

When: Use in agent workflows that can call external tools or APIs.

This notebook demonstrates only the prompt pattern; it does not provide a live search/tool integration.

Example prompt

Plan how you would research the latest EV market trends: list search keywords, describe the search action you would take, explain what evidence you would inspect, and how you would refine the search.

In [ ]:
# Run: ReAct-style planning (no live tool call in this notebook)
PROMPT = r"""Plan how you would research the latest EV market trends.
List search keywords, describe the search action you would take, explain what evidence you would inspect, and how you would refine the search."""
out = run(PROMPT, **GEN_ARGS)
print(out)
log_result("react-style", "example", PROMPT, out)

In [ ]:
# Optional: Validate JSON outputs for structured prompts
def ensure_json(text):
    try:
        return json.loads(text)
    except Exception:
        import re
        m = re.search(r"\{[\s\S]*\}", text)
        if m:
            return json.loads(m.group(0))
        raise